# Engineering feasibility verification

Standalone. Does **not** require any other notebook — the four problem definitions are
reproduced verbatim from `05_Engineering.ipynb` cell 6, so nothing needs re-running.

## What this checks

The engineering experiments used a static squared penalty, `1e6 * sum(max(0, g)**2)`, with
no post-hoc feasibility test. A solution violating a constraint by *v* has `1e6 * v**2`
added to its objective, so a violation of 2x10^-3 costs only 4.0 — small enough that a
marginally infeasible solution can score below the true constrained optimum.

This matters for one claim in particular: QSO's reported best on Speed Reducer Design is
2990.40 against a cited optimum of 2994.47. Either QSO found a better feasible solution, or
it found an infeasible one whose penalty was too small to expose it.

The notebook re-evaluates every algorithm's stored best solution on all four problems and
reports the true objective, the maximum constraint violation, and whether the solution is
feasible at a stated tolerance.

In [1]:
# === 1. Setup ===
from google.colab import drive
drive.mount('/content/drive')

import json, numpy as np, pandas as pd

BASE = '/content/drive/MyDrive/QSO_Research'
RESULTS = f'{BASE}/results/raw/engineering/all_results.json'

FEAS_TOL = 1e-6      # feasibility tolerance: max(0, g_i) must not exceed this

res = json.load(open(RESULTS))
print(f'loaded {len(res)} algorithm-problem entries')
print('keys per entry:', sorted(res[0].keys()))
print('problems:', sorted({r['problem'] for r in res}))
print('algorithms:', len(sorted({r['algo'] for r in res})))

Mounted at /content/drive
loaded 75 algorithm-problem entries
keys per entry: ['algo', 'best', 'best_x', 'mean', 'problem', 'runs', 'std', 'worst']
problems: ['PVD', 'REB', 'SRD', 'TCSD', 'WBD']
algorithms: 15


In [2]:
# === 2. Problem definitions (verbatim from 05_Engineering.ipynb cell 6) ===
# Each returns (raw objective, constraint vector g) so violations can be inspected
# separately from the penalised value the optimiser actually saw.

def wbd_parts(x):
    h, l, t, b = x
    P, L, E, G = 6000, 14, 30e6, 12e6
    t_m, s_m, d_m = 13600, 30000, 0.25
    M = P*(L + l/2)
    R = np.sqrt(l**2/4 + ((h+t)/2)**2)
    J = 2*(np.sqrt(2)*h*l*(l**2/12 + ((h+t)/2)**2))
    t1 = P/(np.sqrt(2)*h*l)
    t2 = M*R/J
    tau = np.sqrt(t1**2 + 2*t1*t2*l/(2*R) + t2**2)
    sigma = 6*P*L/(b*t**2)
    delta = 6*P*L**3/(E*b*t**3)
    Pc = (4.013*E*np.sqrt(t**2*b**6/36)/L**2*(1 - t/(2*L)*np.sqrt(E/(4*G))))
    f = 1.10471*h**2*l + 0.04811*t*b*(14+l)
    g = [tau - t_m, sigma - s_m, h - b,
         0.10471*h**2 + 0.04811*t*b*(14+l) - 5.0,
         0.125 - h, delta - d_m, P - Pc]
    return f, np.array(g)

def pvd_parts(x):
    Ts, Th, R, L = x
    f = 0.6224*Ts*R*L + 1.7781*Th*R**2 + 3.1661*Ts**2*L + 19.84*Ts**2*R
    g = [-Ts + 0.0193*R, -Th + 0.00954*R,
         -np.pi*R**2*L - (4/3)*np.pi*R**3 + 1296000, L - 240]
    return f, np.array(g)

def tcsd_parts(x):
    d, D, N = x
    f = (N + 2)*D*d**2
    g = [1 - D**3*N/(71785*d**4),
         (4*D**2 - d*D)/(12566*(D*d**3 - d**4)) + 1/(5108*d**2) - 1,
         1 - 140.45*d/(D**2*N),
         (D + d)/1.5 - 1]
    return f, np.array(g)

def srd_parts(x):
    b, m, z, l1, l2, d1, d2 = x
    f = (0.7854*b*m**2*(3.3333*z**2 + 14.9334*z - 43.0934)
         - 1.508*b*(d1**2 + d2**2) + 7.4777*(d1**3 + d2**3)
         + 0.7854*(l1*d1**2 + l2*d2**2))
    g = [27/(b*m**2*z) - 1,
         397.5/(b*m**2*z**2) - 1,
         1.93*l1**3/(m*z*d1**4) - 1,
         1.93*l2**3/(m*z*d2**4) - 1,
         np.sqrt((745*l1/(m*z))**2 + 16.9e6)/(110*d1**3) - 1,
         np.sqrt((745*l2/(m*z))**2 + 157.5e6)/(85*d2**3) - 1,
         m*z/40 - 1, 5*m/b - 1, b/(12*m) - 1,
         1.5*d1/l1 - 1, 1.1*d2/l2 - 1]
    return f, np.array(g)

PARTS = {'WBD': wbd_parts, 'PVD': pvd_parts, 'TCSD': tcsd_parts, 'SRD': srd_parts}
OPT   = {'WBD': 1.7248, 'PVD': 5804.45, 'TCSD': 0.012665, 'SRD': 2994.47}
print('four problems defined')

four problems defined


In [3]:
# === 3. Verify every stored best solution ===
rows = []
for r in res:
    p = r['problem']
    if p not in PARTS:            # REB excluded
        continue
    x = np.array(r.get('best_x', []), dtype=float)
    if x.size == 0:
        rows.append(dict(problem=p, algo=r['algo'], note='no solution stored'))
        continue
    try:
        f, g = PARTS[p](x)
    except Exception as e:
        rows.append(dict(problem=p, algo=r['algo'], note=f'eval failed: {e}'))
        continue
    viol = np.maximum(0.0, g)
    rows.append(dict(
        problem=p, algo=r['algo'],
        reported=float(r['best']),
        raw_objective=float(f),
        penalty=float(1e6*np.sum(viol**2)),
        max_violation=float(viol.max()),
        n_violated=int((viol > FEAS_TOL).sum()),
        feasible=bool(viol.max() <= FEAS_TOL),
        beats_optimum=bool(float(r['best']) < OPT[p]),
    ))

df = pd.DataFrame(rows)
pd.set_option('display.width', 200, 'display.max_rows', 100)
print(df.to_string(index=False))

problem algo    reported  raw_objective      penalty  max_violation  n_violated  feasible  beats_optimum
    WBD  QSO    1.737732       1.737732 3.308722e-16   1.818989e-11           0      True          False
    WBD  PSO    1.724852       1.724851 4.686312e-07   6.845664e-07           0      True          False
    WBD   GA    1.771065       1.771065 0.000000e+00   0.000000e+00           0      True          False
    WBD   DE    1.724852       1.724851 4.582133e-07   6.769145e-07           0      True          False
    WBD  GWO    1.726500       1.726500 0.000000e+00   0.000000e+00           0      True          False
    WBD  WOA    1.726693       1.726693 0.000000e+00   0.000000e+00           0      True          False
    WBD  SCA    1.802842       1.802842 0.000000e+00   0.000000e+00           0      True          False
    WBD  HHO    1.768508       1.768508 0.000000e+00   0.000000e+00           0      True          False
    WBD  MPA    1.884231       1.884231 0.000000e+00   

In [4]:
# === 4. The specific claim: QSO on Speed Reducer Design ===
q = df[(df.problem=='SRD') & (df.algo=='QSO')].iloc[0]
srd_entry = [r for r in res if r['problem']=='SRD' and r['algo']=='QSO'][0]
x = np.array(srd_entry['best_x'], dtype=float)
f, g = srd_parts(x)
viol = np.maximum(0.0, g)

print('SPEED REDUCER DESIGN — QSO best solution')
print('-'*58)
for name, val in zip(['b','m','z','l1','l2','d1','d2'], x):
    print(f'  {name:>3} = {val:.6f}')
print()
print(f'  raw objective f(x)      : {f:.4f}')
print(f'  penalty contribution    : {1e6*np.sum(viol**2):.4f}')
print(f'  penalised (as reported) : {f + 1e6*np.sum(viol**2):.4f}')
print(f'  value stored in results : {srd_entry["best"]:.4f}')
print(f'  cited optimum           : {OPT["SRD"]:.4f}')
print()
print('  constraint violations:')
for i, v in enumerate(viol, 1):
    flag = '  <-- VIOLATED' if v > FEAS_TOL else ''
    print(f'    g{i:<3} = {g[i-1]:+.6e}   violation {v:.3e}{flag}')
print()
print(f'  max violation : {viol.max():.6e}')
print(f'  FEASIBLE at tol {FEAS_TOL:.0e}: {viol.max() <= FEAS_TOL}')

SPEED REDUCER DESIGN — QSO best solution
----------------------------------------------------------
    b = 3.497583
    m = 0.700000
    z = 17.000484
   l1 = 7.874174
   l2 = 7.314087
   d1 = 3.351549
   d2 = 5.284955

  raw objective f(x)      : 2989.1304
  penalty contribution    : 1.2692
  penalised (as reported) : 2990.3997
  value stored in results : 2990.3997
  cited optimum           : 2994.4700

  constraint violations:
    g1   = -7.330180e-02   violation 0.000e+00
    g2   = -1.974901e-01   violation 0.000e+00
    g3   = -3.724754e-01   violation 0.000e+00
    g4   = -9.186583e-01   violation 0.000e+00
    g5   = -1.980906e-04   violation 0.000e+00
    g6   = +8.897270e-04   violation 8.897e-04  <-- VIOLATED
    g7   = -7.024915e-01   violation 0.000e+00
    g8   = +6.911004e-04   violation 6.911e-04  <-- VIOLATED
    g9   = -5.836211e-01   violation 0.000e+00
    g10  = -3.615428e-01   violation 0.000e+00
    g11  = -2.051707e-01   violation 0.000e+00

  max violation : 8.

In [5]:
# === 5. Verdict ===
mv = float(np.maximum(0.0, srd_parts(np.array(srd_entry['best_x'], float))[1]).max())

print('INTERPRETATION\n')
if mv <= FEAS_TOL:
    print(f'  Max violation {mv:.2e} is within tolerance {FEAS_TOL:.0e}.')
    print('  The solution is FEASIBLE and the improvement on the cited optimum is genuine.')
    print('  -> Keep the claim in Table 12 and state this verified tolerance in the footnote.')
else:
    print(f'  Max violation {mv:.2e} EXCEEDS tolerance {FEAS_TOL:.0e}.')
    print('  The reported value undercuts the cited optimum because the squared penalty')
    print('  (1e6 * v**2) is too small at this magnitude to expose the violation.')
    print('  -> Remove the better-than-optimum claim from Table 12.')
    print('  -> Report instead that the static penalty admits marginally infeasible')
    print('     solutions, which motivates the adaptive penalty proposed in Section 6.4.')
print()
print('ACROSS ALL FOUR PROBLEMS')
summ = df.groupby('problem').agg(
    n_feasible=('feasible','sum'), n_algorithms=('algo','count'),
    n_beating_optimum=('beats_optimum','sum'), worst_violation=('max_violation','max'))
print(summ.to_string())
print()
print('Algorithms reporting a value below the cited optimum:')
bad = df[df.beats_optimum][['problem','algo','reported','max_violation','feasible']]
print(bad.to_string(index=False) if len(bad) else '  none')

INTERPRETATION

  Max violation 8.90e-04 EXCEEDS tolerance 1e-06.
  The reported value undercuts the cited optimum because the squared penalty
  (1e6 * v**2) is too small at this magnitude to expose the violation.
  -> Remove the better-than-optimum claim from Table 12.
  -> Report instead that the static penalty admits marginally infeasible
     solutions, which motivates the adaptive penalty proposed in Section 6.4.

ACROSS ALL FOUR PROBLEMS
         n_feasible  n_algorithms  n_beating_optimum  worst_violation
problem                                                              
PVD               2            15                  0     1.164694e-02
SRD               0            15                 13     3.809327e-03
TCSD             15            15                  0     3.231252e-08
WBD              15            15                  0     6.845664e-07

Algorithms reporting a value below the cited optimum:
problem algo    reported  max_violation  feasible
    SRD  QSO 2990.399670   

In [6]:
# === 6. Save for the manuscript ===
out = f'{BASE}/results/raw/engineering/feasibility_verification.csv'
df.to_csv(out, index=False)
print('saved:', out)
print()
print('If any solution is infeasible, this file is also worth adding to the repository')
print('cited in Section 4.5 — it documents the constraint-handling limitation directly.')

saved: /content/drive/MyDrive/QSO_Research/results/raw/engineering/feasibility_verification.csv

If any solution is infeasible, this file is also worth adding to the repository
cited in Section 4.5 — it documents the constraint-handling limitation directly.
